## Tips

- **Start simple** — a single GRU with 64 units is a good first attempt.
  Only add complexity (stacking, dropout, bidirectional) if the simple model
  is clearly underfitting or overfitting.

- **Use many epochs** — RNNs on time series converge slowly. 10 or 20 epochs
  is almost never enough. Use **at least 100 epochs** and let early stopping
  decide when to stop. The best checkpoint is saved automatically by
  `ModelCheckpoint` — you will not miss the optimal point even if you
  train for too long.

- **Watch the train/val gap** — if train MAE is much lower than val MAE
  you are overfitting. Add dropout, reduce `hidden_size`, or increase
  early stopping patience to give regularization more time to work.

- **Use TensorBoard** — run `tensorboard --logdir runs` in your terminal
  to monitor train and val curves in real time. A healthy training curve
  shows both losses decreasing together. A diverging val curve means overfitting.

- **Learning rate matters** — if the loss is not decreasing after the first
  few epochs, try a lower learning rate (`1e-4` instead of `1e-3`).
  Use `ReduceLROnPlateau` to automatically decay the learning rate when
  val MAE stops improving.

- **The sklearn baseline is hard to beat** — `HistGradientBoosting` with
  explicit lag features is a very strong baseline for tabular time series.
  This is not a failure of the RNN — it reflects a fundamental difference
  between the two approaches: tree models get temporal information from
  hand-crafted features, while RNNs must learn it from raw sequences.
  Getting within 10 bikes/hour of the sklearn baseline (< 44 bikes/hour)
  is an excellent result.

In [ ]:
import os
import numpy as np
import joblib

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

In [29]:
save_dir = "data/bike_processed"

# --- Load arrays ---
raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
counts = np.load(os.path.join(save_dir, "counts.npy"))
count_stats = np.load(os.path.join(save_dir, "count_stats.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
naive_mae   = np.load(os.path.join(save_dir, "naive_mae.npy"))
preprocessor = joblib.load(os.path.join(save_dir, "preprocessor_rnn.pkl"))

# --- Recover split sizes ---
num_train = len(train_idx)
num_val   = len(val_idx)
num_test  = len(test_idx)

# --- Count stats in the training set ---
count_mean  = count_stats[0]
count_std   = count_stats[1]

# --- Sanity check ---
print(f"raw_data shape:    {raw_data.shape}")
print(f"counts shape: {counts.shape}")
print(f"train/val/test:    {num_train} / {num_val} / {num_test}")
print(f"counts range: {counts.min():.0f} to {counts.max():.0f} bikes/hour")
print(f"\nNaive baseline — Val MAE: {naive_mae[0]:.2f} | Test MAE: {naive_mae[1]:.2f} bikes/hour")

raw_data shape:    (17210, 28)
counts shape: (17210,)
train/val/test:    12047 / 2581 / 2582
counts range: 1 to 977 bikes/hour

Naive baseline — Val MAE: 93.26 | Test MAE: 80.78 bikes/hour


In [23]:
count_norm = (counts - count_mean) / count_std

In [6]:
class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [22]:
sampling_rate   = 1    # already hourly
sequence_length = 24   # look back 24 hours
delay           = 1    # predict 1 hour ahead
batch_size      = 256

train_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train,
)
val_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train, end_index=num_train + num_val,
)
test_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train + num_val, end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# --- Sanity check ---
inputs, targets = next(iter(train_loader))
print(f"Input shape:  {inputs.shape}")   # (256, 24, num_features)
print(f"Target shape: {targets.shape}")  # (256,)
print(f"Target range: {targets.min():.0f} to {targets.max():.0f}")

Input shape:  torch.Size([256, 24, 28])
Target shape: torch.Size([256])
Target range: -1 to 4


## Desarrollo — Modelo recurrente para la demanda de bicicletas

A partir de aquí se construye el modelo siguiendo los *Tips* del inicio: una GRU simple, entrenamiento largo con early stopping y guardado del mejor checkpoint (`ModelCheckpoint`), monitoreo en TensorBoard, `ReduceLROnPlateau` y comparación contra el baseline.

> **Nota sobre los datos.** El `test_dataset` original usa `end_index = len(raw_data) - delay`. Como cada ventana necesita `sequence_length` pasos por delante, los últimos índices de inicio se salen del arreglo y provocan un `IndexError` al iterar el `test_loader`. La primera celda de esta sección lo reconstruye de forma segura, sin modificar el código original.

In [ ]:
# --- Corrección del test_loader ---
# El test_dataset original usa end_index = len(raw_data) - delay. Como cada
# ventana necesita `sequence_length` pasos por delante, los últimos índices de
# inicio se salen del arreglo y provocan un IndexError al iterar el test_loader.
# Lo reconstruimos con un end_index que garantiza que cada ventana cabe completa.
safe_end = len(raw_data) - sequence_length

test_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train + num_val, end_index=safe_end,
)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

xb, yb = next(iter(test_loader))
print(f"test_dataset seguro: {len(test_dataset)} ventanas")
print(f"Batch de test: x={tuple(xb.shape)}  y={tuple(yb.shape)}")

In [ ]:
class BikeRNN(nn.Module):
    """GRU para predecir la demanda horaria de bicicletas (regresión)."""

    def __init__(self, n_features, hidden_size=64, num_layers=1, dropout=0.0):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, sequence_length, n_features)
        out, _ = self.gru(x)
        last = out[:, -1, :]                  # estado oculto del último paso
        return self.head(last).squeeze(-1)    # (batch,)

In [ ]:
# --- Configuración del entrenamiento ---
import matplotlib.pyplot as plt
from datetime import datetime

torch.manual_seed(42)

device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_features = raw_data.shape[1]

model     = BikeRNN(n_features, hidden_size=64, num_layers=1).to(device)
criterion = nn.L1Loss()                                    # MAE en escala normalizada
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5,
)

print(f"Dispositivo: {device}")
print(model)
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters()):,}")


@torch.no_grad()
def evaluate(model, loader):
    """MAE sobre un loader, devuelto en bicicletas/hora (escala original)."""
    model.eval()
    abs_err, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        abs_err += torch.abs(pred - y).sum().item()
        n += y.numel()
    return (abs_err / n) * count_std

In [ ]:
# --- Bucle de entrenamiento ---
EPOCHS   = 100
PATIENCE = 15                       # paciencia del early stopping
CKPT     = "best_model.pt"

run_dir = os.path.join("runs", "bike_gru_" + datetime.now().strftime("%Y%m%d-%H%M%S"))
writer  = SummaryWriter(run_dir)

best_val_mae      = float("inf")
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    # --- entrenamiento ---
    model.train()
    abs_err, n = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        abs_err += torch.abs(pred - y).sum().item()
        n += y.numel()
    train_mae = (abs_err / n) * count_std

    # --- validación ---
    val_mae = evaluate(model, val_loader)
    scheduler.step(val_mae)
    lr_now = optimizer.param_groups[0]["lr"]

    # --- TensorBoard ---
    writer.add_scalar("MAE/train", train_mae, epoch)
    writer.add_scalar("MAE/val",   val_mae,   epoch)
    writer.add_scalar("lr",        lr_now,    epoch)

    # --- ModelCheckpoint + early stopping ---
    if val_mae < best_val_mae - 1e-4:
        best_val_mae      = val_mae
        epochs_no_improve = 0
        torch.save(model.state_dict(), CKPT)
        mejor = "  <-- mejor checkpoint"
    else:
        epochs_no_improve += 1
        mejor = ""

    if epoch % 5 == 0 or mejor:
        print(f"Epoca {epoch:3d} | train MAE {train_mae:6.2f} | "
              f"val MAE {val_mae:6.2f} | lr {lr_now:.1e}{mejor}")

    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping en la epoca {epoch} "
              f"(sin mejora en {PATIENCE} epocas).")
        break

writer.close()
print(f"\nMejor val MAE: {best_val_mae:.2f} bicicletas/hora")

In [ ]:
# Cargar el mejor checkpoint guardado durante el entrenamiento
model.load_state_dict(torch.load(CKPT))

val_mae  = evaluate(model, val_loader)
test_mae = evaluate(model, test_loader)

print("Modelo GRU — mejor checkpoint\n")
print(f"{'Modelo':<24}{'Val MAE':>10}{'Test MAE':>11}")
print("-" * 45)
print(f"{'Baseline ingenuo':<24}{naive_mae[0]:>10.2f}{naive_mae[1]:>11.2f}")
print(f"{'GRU (este modelo)':<24}{val_mae:>10.2f}{test_mae:>11.2f}")

In [ ]:
# --- Predicciones vs valores reales en el conjunto de test ---
model.eval()
preds, trues = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.append(model(x.to(device)).cpu())
        trues.append(y)

# de escala normalizada de vuelta a bicicletas/hora
preds = torch.cat(preds).numpy() * count_std + count_mean
trues = torch.cat(trues).numpy() * count_std + count_mean

plt.figure(figsize=(14, 4))
plt.plot(trues[:300], label="Real",           linewidth=1.5)
plt.plot(preds[:300], label="Predicción GRU", linewidth=1.5, alpha=0.8)
plt.title("Demanda de bicicletas — conjunto de test (primeras 300 horas)")
plt.xlabel("Hora"); plt.ylabel("Bicicletas/hora")
plt.legend(); plt.tight_layout(); plt.show()

## Análisis de resultados

- **Comparación con el baseline ingenuo.** El baseline ingenuo (repetir el valor de hace 24 h) tiene un MAE de ~93 (val) y ~81 (test) bicicletas/hora. La GRU debería reducir claramente ese error al aprender las dependencias temporales de la serie.
- **Brecha train/val.** Si el MAE de entrenamiento queda muy por debajo del de validación, el modelo sobreajusta: conviene añadir `dropout`, reducir `hidden_size` o aumentar la paciencia del early stopping. Las curvas de TensorBoard (`tensorboard --logdir runs`) permiten diagnosticarlo.
- **`ModelCheckpoint` y early stopping.** Se guarda el mejor estado según el MAE de validación, así que entrenar de más no degrada el resultado final: siempre se recupera el mejor checkpoint.
- **Baseline de sklearn.** Según los *Tips*, `HistGradientBoosting` con features de lag explícitas es un baseline muy fuerte (MAE < 44). Acercarse a ~10 bicicletas/hora de esa cifra es un resultado excelente para la GRU: el modelo de árboles recibe la información temporal ya construida como features, mientras que la GRU debe aprenderla desde las secuencias crudas.

> Completa esta sección con los valores concretos de MAE que obtengas al ejecutar el notebook en Lightning AI.